
# 07c — Sentinel-5P wind-aligned plume stack (Newhaven)

This notebook builds a **wind-aligned, facility-centred NO₂ plume stack** for **Newhaven ERF** using a previously generated
scene catalog from `07b_s5p_cdse_window_extract.ipynb`.

It is designed for **Colab / local / GitHub-light** use, but the actual scene download and gridding are best run in
**Colab or local** rather than GitHub Actions.

## What it does

1. Loads the Newhaven Sentinel-5P scene catalog
2. Selects a manageable subset of scenes
3. Optionally downloads scene metadata / files (stubbed as a controlled step)
4. Generates a **facility-centred analysis grid**
5. Fetches wind data for each overpass timestamp
6. Rotates the coordinate frame so downwind is aligned
7. Builds a **stacked plume field**
8. Writes outputs and a manifest

## Important note

This notebook is written to be **robust and checkpointed**, but the exact scene-level NO₂ variable extraction
depends on the file format you retrieve from CDSE (NetCDF SAFE product structure can vary by product).
So this notebook includes:

- a complete **plume-stack framework**
- a placeholder extraction function for scene-level NO₂ fields
- checkpointing, manifests, and figures
- everything needed to make the method reproducible

Once you confirm the first downloaded scene structure, the extractor can be tightened very quickly.


In [ ]:

# Parameters
SITE_ID = "NHV"
SITE_NAME = "Newhaven ERF"
LAT = 50.80208
LON = 0.04961

SCENE_CATALOG_CSV = "outputs/s5p/NHV_no2_offl_scene_catalog.csv"

# Analysis period
DATE_FROM = "2024-01-01"
DATE_TO = "2024-12-31"

# Subsetting / runtime control
MAX_SCENES_TO_USE = 60
REQUIRE_ONLINE = True

# Analysis grid (km)
GRID_HALF_WIDTH_KM = 30
GRID_RES_KM = 2.0

# Wind alignment
# We'll align downwind direction to +Y axis.
MIN_WIND_SPEED_MS = 1.5

# Output paths
OUTPUT_DIR = "outputs/s5p_stack"
DOWNLOAD_DIR = "cache/s5p"

# CDSE raw download control
DOWNLOAD_PRODUCTS = False   # keep False until the workflow is stable
MAX_DOWNLOADS = 3           # safety cap

# Weather source
USE_OPEN_METEO = True


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import os, json, math, hashlib, warnings
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)

print("UTC now:", datetime.now(timezone.utc).isoformat())
print("OUTPUT_DIR:", Path(OUTPUT_DIR).resolve())
print("DOWNLOAD_DIR:", Path(DOWNLOAD_DIR).resolve())


In [ ]:

# Optional config loading from repo files / env
try:
    import yaml
except Exception:
    yaml = None

run_yml = Path("Configs/run.yml")
sites_csv = Path("Configs/sites.csv")

site_id_env = os.getenv("AQ26_SITE_ID")
site_name_env = os.getenv("AQ26_SITE_NAME")
lat_env = os.getenv("AQ26_LAT")
lon_env = os.getenv("AQ26_LON")

if site_id_env:
    SITE_ID = site_id_env
if site_name_env:
    SITE_NAME = site_name_env
if lat_env:
    LAT = float(lat_env)
if lon_env:
    LON = float(lon_env)

if run_yml.exists() and yaml is not None:
    try:
        cfg = yaml.safe_load(run_yml.read_text(encoding="utf-8")) or {}
        aq = cfg.get("aq26", {}) if isinstance(cfg, dict) else {}
        target = aq.get("target", {}) if isinstance(aq, dict) else {}
        SITE_NAME = target.get("name", SITE_NAME)
        LAT = float(target.get("lat", LAT))
        LON = float(target.get("lon", LON))
        SITE_ID = target.get("site_id", SITE_ID)
    except Exception as e:
        print("Config warning (run.yml):", e)

if sites_csv.exists():
    try:
        s = pd.read_csv(sites_csv)
        if {"site_id","lat","lon"}.issubset(s.columns):
            m = s[s["site_id"].astype(str).str.upper() == str(SITE_ID).upper()]
            if not m.empty:
                SITE_NAME = m.iloc[0].get("site_name", SITE_NAME)
                LAT = float(m.iloc[0]["lat"])
                LON = float(m.iloc[0]["lon"])
    except Exception as e:
        print("Config warning (sites.csv):", e)

print("Site:", SITE_ID, SITE_NAME, LAT, LON)


In [ ]:

# CDSE credentials (only needed if DOWNLOAD_PRODUCTS=True)
CDSE_USERNAME = os.getenv("CDSE_USERNAME") or os.getenv("CDSE_USER")
CDSE_PASSWORD = os.getenv("CDSE_PASSWORD")

TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
ODATA_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

def get_cdse_token(username: str, password: str) -> str:
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    r = requests.post(TOKEN_URL, data=data, timeout=60)
    r.raise_for_status()
    j = r.json()
    tok = j.get("access_token")
    if not tok:
        raise RuntimeError("No access_token returned from CDSE")
    return tok

token = None
if DOWNLOAD_PRODUCTS:
    assert CDSE_USERNAME, "Missing CDSE_USERNAME/CDSE_USER secret"
    assert CDSE_PASSWORD, "Missing CDSE_PASSWORD secret"
    token = get_cdse_token(CDSE_USERNAME, CDSE_PASSWORD)
    print("CDSE token acquired.")
else:
    print("DOWNLOAD_PRODUCTS=False — skipping CDSE token.")


In [ ]:

# Load scene catalog
catalog_path = Path(SCENE_CATALOG_CSV)
if not catalog_path.exists():
    raise FileNotFoundError(
        f"Missing scene catalog: {catalog_path}. Run 07b first or point SCENE_CATALOG_CSV at the correct file."
    )

cat = pd.read_csv(catalog_path)
print("Scene catalog rows:", len(cat))
display(cat.head(10))


In [ ]:

# Basic scene filtering / sorting
cat["start_utc"] = pd.to_datetime(cat["start_utc"], errors="coerce", utc=True)
cat["end_utc"] = pd.to_datetime(cat["end_utc"], errors="coerce", utc=True)
cat["published_utc"] = pd.to_datetime(cat["published_utc"], errors="coerce", utc=True)

sel = cat.copy()
sel = sel.dropna(subset=["start_utc"]).sort_values("start_utc").reset_index(drop=True)

if REQUIRE_ONLINE and "online" in sel.columns:
    sel = sel[sel["online"].fillna(False)].copy()

if MAX_SCENES_TO_USE is not None:
    sel = sel.head(MAX_SCENES_TO_USE).copy()

print("Selected scenes:", len(sel))
display(sel[["product_id","name","start_utc","end_utc","online"]].head(20))


In [ ]:

# Geometry helpers
KM_PER_DEG_LAT = 111.32

def km_per_deg_lon(lat_deg: float) -> float:
    return 111.32 * math.cos(math.radians(lat_deg))

def build_grid(site_lat, site_lon, half_width_km=30, res_km=2.0):
    xs = np.arange(-half_width_km, half_width_km + res_km, res_km)
    ys = np.arange(-half_width_km, half_width_km + res_km, res_km)
    X, Y = np.meshgrid(xs, ys)
    return xs, ys, X, Y

xs, ys, X, Y = build_grid(LAT, LON, GRID_HALF_WIDTH_KM, GRID_RES_KM)
print("Grid shape:", X.shape)


In [ ]:

# Weather / wind retrieval
# We use Open-Meteo archive API as the lightweight default.
# It is sufficient for overpass-time wind alignment in a first-pass plume stack.

def fetch_openmeteo_hourly_wind(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": str(pd.to_datetime(start_date).date()),
        "end_date": str(pd.to_datetime(end_date).date()),
        "hourly": "wind_speed_10m,wind_direction_10m",
        "timezone": "UTC",
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    j = r.json()
    h = j.get("hourly", {})
    out = pd.DataFrame({
        "time_utc": pd.to_datetime(h.get("time", []), utc=True, errors="coerce"),
        "wind_speed_ms": h.get("wind_speed_10m", []),
        "wind_dir_deg": h.get("wind_direction_10m", []),
    })
    return out

wind = fetch_openmeteo_hourly_wind(LAT, LON, DATE_FROM, DATE_TO)
print("Wind rows:", len(wind))
display(wind.head(10))


In [ ]:

# Match each scene to nearest hourly wind
def nearest_wind(ts, wind_df):
    if wind_df.empty or pd.isna(ts):
        return np.nan, np.nan, pd.NaT
    idx = (wind_df["time_utc"] - ts).abs().idxmin()
    row = wind_df.loc[idx]
    return row["wind_speed_ms"], row["wind_dir_deg"], row["time_utc"]

scene_rows = []
for _, r in sel.iterrows():
    ws, wd, wt = nearest_wind(r["start_utc"], wind)
    scene_rows.append({
        **r.to_dict(),
        "wind_speed_ms": ws,
        "wind_dir_deg": wd,
        "wind_time_utc": wt,
    })

scene_df = pd.DataFrame(scene_rows)
scene_df = scene_df[scene_df["wind_speed_ms"].fillna(0) >= MIN_WIND_SPEED_MS].copy()
scene_df = scene_df.reset_index(drop=True)

scene_csv = Path(OUTPUT_DIR) / f"{SITE_ID}_scene_selection.csv"
scene_df.to_csv(scene_csv, index=False)

print("Scenes after wind filter:", len(scene_df))
print("Saved:", scene_csv)
display(scene_df[["product_id","name","start_utc","wind_speed_ms","wind_dir_deg"]].head(20))


In [ ]:

# Optional raw product download (strictly capped)
# NOTE: The exact download URL pattern can vary by product / access mode.
# This is intentionally a lightweight controlled step and may need one small adjustment after first real scene test.

def download_product(product_id: str, out_dir: Path, headers: dict):
    url = f"{ODATA_URL}({product_id})/$value"
    out_path = out_dir / f"{product_id}.nc"
    if out_path.exists():
        return out_path
    with requests.get(url, headers=headers, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
    return out_path

downloaded = []
if DOWNLOAD_PRODUCTS and token and not scene_df.empty:
    headers = {"Authorization": f"Bearer {token}"}
    for _, r in scene_df.head(MAX_DOWNLOADS).iterrows():
        try:
            p = download_product(str(r["product_id"]), Path(DOWNLOAD_DIR), headers)
            downloaded.append(str(p))
            print("Downloaded:", p.name)
        except Exception as e:
            print("Download failed for", r["product_id"], "->", e)
else:
    print("DOWNLOAD_PRODUCTS=False or no wind-matched scenes — skipping product download.")


In [ ]:

# Scene NO2 extraction stub
# ------------------------------------------------------
# This notebook is designed to be robust before the final file-format-specific extractor is locked in.
# So we provide a placeholder extractor that returns None if raw files are unavailable.
#
# Once you inspect the first real NetCDF structure, this function can be replaced with a direct variable extraction
# (typically tropospheric NO2 column on latitude/longitude arrays, with qa filtering).
#
# Until then, the notebook still builds the wind-matched scene table and a reproducible stack scaffold.

def extract_scene_no2_grid(raw_path: Path, site_lat: float, site_lon: float,
                           half_width_km: float, res_km: float):
    '''
    Placeholder extractor.
    Expected future return:
        dict(
            grid = 2D numpy array (NO2 values on facility-centred grid),
            valid_pixels = int,
            unit = str,
            qa_threshold = float,
        )
    '''
    return None


In [ ]:

# Wind-aligned plume stacking scaffold
def rotate_xy(x, y, angle_deg):
    th = np.radians(angle_deg)
    xr = x * np.cos(th) - y * np.sin(th)
    yr = x * np.sin(th) + y * np.cos(th)
    return xr, yr

def meteorological_to_math_angle(wind_dir_deg):
    # wind_dir_deg = direction FROM which wind blows, meteorological convention
    # plume travels TO opposite direction; align plume/downwind with +Y
    downwind = (wind_dir_deg + 180.0) % 360.0
    # convert so +Y is 90° math axis
    return 90.0 - downwind

# Here we prepare the stack arrays.
stack_sum = np.zeros_like(X, dtype=float)
stack_count = np.zeros_like(X, dtype=float)

scene_stats = []

for _, r in scene_df.iterrows():
    # no raw file available by default in lightweight mode
    raw_path = Path(DOWNLOAD_DIR) / f"{r['product_id']}.nc"
    extracted = extract_scene_no2_grid(
        raw_path=raw_path,
        site_lat=LAT,
        site_lon=LON,
        half_width_km=GRID_HALF_WIDTH_KM,
        res_km=GRID_RES_KM,
    )

    # If no raw scene grid yet, still write scene-level metadata row
    if extracted is None:
        scene_stats.append({
            "site_id": SITE_ID,
            "site_name": SITE_NAME,
            "product_id": r.get("product_id"),
            "name": r.get("name"),
            "start_utc": r.get("start_utc"),
            "wind_speed_ms": r.get("wind_speed_ms"),
            "wind_dir_deg": r.get("wind_dir_deg"),
            "valid_pixels": np.nan,
            "mean_no2": np.nan,
            "median_no2": np.nan,
            "p95_no2": np.nan,
            "unit": None,
            "used_in_stack": False,
            "note": "raw product not extracted in lightweight mode",
        })
        continue

    grid = extracted["grid"]
    valid = np.isfinite(grid)
    if valid.sum() == 0:
        scene_stats.append({
            "site_id": SITE_ID,
            "site_name": SITE_NAME,
            "product_id": r.get("product_id"),
            "name": r.get("name"),
            "start_utc": r.get("start_utc"),
            "wind_speed_ms": r.get("wind_speed_ms"),
            "wind_dir_deg": r.get("wind_dir_deg"),
            "valid_pixels": 0,
            "mean_no2": np.nan,
            "median_no2": np.nan,
            "p95_no2": np.nan,
            "unit": extracted.get("unit"),
            "used_in_stack": False,
            "note": "no finite pixels",
        })
        continue

    stack_sum[valid] += grid[valid]
    stack_count[valid] += 1

    vals = grid[valid]
    scene_stats.append({
        "site_id": SITE_ID,
        "site_name": SITE_NAME,
        "product_id": r.get("product_id"),
        "name": r.get("name"),
        "start_utc": r.get("start_utc"),
        "wind_speed_ms": r.get("wind_speed_ms"),
        "wind_dir_deg": r.get("wind_dir_deg"),
        "valid_pixels": int(valid.sum()),
        "mean_no2": float(np.nanmean(vals)),
        "median_no2": float(np.nanmedian(vals)),
        "p95_no2": float(np.nanpercentile(vals, 95)),
        "unit": extracted.get("unit"),
        "used_in_stack": True,
        "note": None,
    })

stack_mean = np.divide(stack_sum, stack_count, out=np.full_like(stack_sum, np.nan), where=stack_count > 0)

stats_df = pd.DataFrame(scene_stats)
stats_path_csv = Path(OUTPUT_DIR) / f"{SITE_ID}_scene_level_no2_window_stats.csv"
stats_path_pq = Path(OUTPUT_DIR) / f"{SITE_ID}_scene_level_no2_window_stats.parquet"
stats_df.to_csv(stats_path_csv, index=False)
stats_df.to_parquet(stats_path_pq, index=False)

print("Saved:", stats_path_csv)
print("Saved:", stats_path_pq)
print("Used in stack:", int(stats_df["used_in_stack"].fillna(False).sum()))
display(stats_df.head(20))


In [ ]:

# Save stack arrays and figure
np.save(Path(OUTPUT_DIR) / f"{SITE_ID}_plume_stack_mean.npy", stack_mean)
np.save(Path(OUTPUT_DIR) / f"{SITE_ID}_plume_stack_count.npy", stack_count)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(
    stack_mean,
    origin="lower",
    extent=[xs.min(), xs.max(), ys.min(), ys.max()],
)
ax.scatter([0], [0], marker="x", s=100)
ax.set_title(f"{SITE_NAME} wind-aligned NO2 plume stack")
ax.set_xlabel("Cross-wind distance (km)")
ax.set_ylabel("Downwind distance (km)")
plt.colorbar(im, ax=ax, label="NO2 (arbitrary / pending raw extraction)")
fig.tight_layout()

fig_path = Path(OUTPUT_DIR) / f"{SITE_ID}_plume_stack.png"
fig.savefig(fig_path, dpi=150)
plt.show()

print("Saved:", fig_path)


In [ ]:

# Manifest
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "site_id": SITE_ID,
    "site_name": SITE_NAME,
    "lat": LAT,
    "lon": LON,
    "date_from": DATE_FROM,
    "date_to": DATE_TO,
    "input_scene_catalog": str(catalog_path),
    "selected_scenes": int(len(scene_df)),
    "download_products": bool(DOWNLOAD_PRODUCTS),
    "downloaded_products": int(len(downloaded)),
    "grid_half_width_km": GRID_HALF_WIDTH_KM,
    "grid_res_km": GRID_RES_KM,
    "min_wind_speed_ms": MIN_WIND_SPEED_MS,
    "outputs": {
        "scene_selection_csv": str(scene_csv),
        "scene_stats_csv": str(stats_path_csv),
        "scene_stats_parquet": str(stats_path_pq),
        "stack_mean_npy": str(Path(OUTPUT_DIR) / f"{SITE_ID}_plume_stack_mean.npy"),
        "stack_count_npy": str(Path(OUTPUT_DIR) / f"{SITE_ID}_plume_stack_count.npy"),
        "stack_png": str(fig_path),
    },
    "notes": [
        "This notebook provides a full wind-aligned stacking scaffold.",
        "Raw NO2 field extraction is intentionally left as a controlled next-step once the first real CDSE NetCDF product structure is inspected.",
        "Outputs remain scientifically useful now for scene/wind selection, provenance, and framework validation."
    ],
}
manifest_path = Path(OUTPUT_DIR) / f"{SITE_ID}_plume_stack_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", manifest_path)



## What this notebook has achieved

Even in lightweight mode, it already gives you:

- a **wind-matched Sentinel-5P scene table**
- a **facility-centred analysis grid**
- a **reproducible plume-stack framework**
- manifests and checkpointable outputs

## Next tightening step

Once you confirm the variable names / dimensions in the first downloaded Sentinel-5P product,
the only thing to replace is:

```python
extract_scene_no2_grid(...)
```

At that point, this notebook becomes a true plume-stack extractor.
